# 05 · The set prediction loss: Hungarian matching

> **Paper:** §3.1, Equations (1) and (2)
>
> **This notebook is self-contained.** The Hungarian algorithm, the cost matrix, the
> set loss and even DETR itself are all written out below — nothing is imported from
> this repo. Compare with [`models/matcher.py`](../models/matcher.py) and
> [`SetCriterion`](../models/detr.py#L110) when you are done.

This is the idea that makes DETR work. The architecture is ordinary; **the loss is the invention.**

**The problem.** The model outputs 100 predictions in a fixed order. The image contains 5 objects in *no* order. How do you compute a loss between an ordered list and an unordered set?

Get this wrong and the model can't learn at all. Get it right and NMS becomes unnecessary.

> **Heads-up on one idiom.** `SetCriterion` builds its targets with *advanced indexing* — `target_classes[idx] = target_classes_o`, where `idx` is a tuple of two tensors. It is the least obvious line in the repo. [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) §4 walks through it with printable 5-query tensors if you want the mechanics first.

## 0. Setup — everything this notebook needs

Only third-party packages are imported. Every DETR-specific piece is defined in this notebook, so you can read top-to-bottom and never wonder what a helper does.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

Boxes come in two conventions and it is a constant source of bugs:

- **`cxcywh`** — centre + size, normalized to `[0, 1]`. This is what DETR predicts.
- **`xyxy`** — two corners. This is what IoU and plotting want.

`generalized_box_iou` is the one worth reading closely; it is what makes the box cost keep working when two boxes do not overlap at all.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

## 1. Why the obvious approaches fail

**Idea A — fix an order** (e.g. sort ground truth left-to-right, compare position by position).
Now prediction #3 is punished for being correct-but-in-the-wrong-slot. The model wastes capacity learning an arbitrary sorting convention, and one extra object at the left edge shifts every target.

**Idea B — match each prediction to its nearest ground-truth box.**
Nothing stops *ten* predictions from all choosing the same cat. They all get rewarded, so the model happily emits ten copies — and now you need NMS again. This is exactly the duplicate problem.

**DETR's answer — a bipartite matching.** Find the *one-to-one* assignment of predictions to ground truths with the lowest total cost. One-to-one is the key: once a prediction claims the cat, no other prediction can, so duplicates are actively penalized.

> *"Our loss produces an optimal bipartite matching between predicted and ground truth objects."* — §3.1

Finding the cheapest one-to-one assignment is a classic problem, solved exactly by the **Hungarian algorithm** in O(n³). Most code reaches for `scipy.optimize.linear_sum_assignment`; in §3 below we write the algorithm out instead, so there is no black box left in the loss.

## 2. A toy example you can check by hand

3 predictions, 2 ground-truth objects. Boxes are `(cx, cy, w, h)`, normalized.

In [ ]:
# ground truth: a cat on the left, a remote on the right
gt_boxes  = torch.tensor([[0.25, 0.50, 0.30, 0.40],     # object 0
                          [0.75, 0.40, 0.20, 0.20]])    # object 1
gt_labels = torch.tensor([17, 75])                       # 17=cat, 75=remote

# predictions: #0 near the cat, #1 also near the cat (a duplicate!), #2 near the remote
pred_boxes = torch.tensor([[0.27, 0.52, 0.28, 0.38],
                           [0.30, 0.48, 0.34, 0.44],
                           [0.72, 0.42, 0.22, 0.19]])

# class probabilities: pretend the model is fairly confident
pred_probs = torch.full((3, 92), 0.001)
pred_probs[0, 17] = 0.90;  pred_probs[1, 17] = 0.70;  pred_probs[2, 75] = 0.85
pred_probs = pred_probs / pred_probs.sum(-1, keepdim=True)

print("3 predictions vs 2 ground-truth objects")
print("note predictions 0 and 1 are BOTH sitting on the cat")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for i, b in enumerate(box_cxcywh_to_xyxy(gt_boxes)):
    ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                 fill=False, color="black", lw=3))
    ax.text(b[0], b[1]-0.02, f"GT {i} ({COCO_CLASSES[gt_labels[i]]})", fontsize=10)
for i, b in enumerate(box_cxcywh_to_xyxy(pred_boxes)):
    c = COLORS[i]
    ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                 fill=False, color=c, lw=2, ls="--"))
    ax.text(b[2], b[3]+0.03, f"pred {i}", color=c, fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_title("solid = ground truth,  dashed = predictions")
plt.show()

### The cost matrix

For every (prediction, ground-truth) pair, DETR computes three costs and adds them with the paper's default weights (§4, `--set_cost_*`): **class 1, L1 5, GIoU 2**.

| Term | Formula | Code |
|---|---|---|
| class | `-p̂(correct class)` | `cost_class = -out_prob[:, tgt_ids]` |
| L1 | `‖b_pred − b_gt‖₁` | `torch.cdist(out_bbox, tgt_bbox, p=1)` |
| GIoU | `−GIoU(b_pred, b_gt)` | `-generalized_box_iou(...)` |

(the `Code` column names the functions we defined in §0)

Note the class term uses the **probability**, not log-probability. The paper explains: *"we use probabilities... This makes the class prediction term commensurable to the box loss"* (§3.1).

In [ ]:
COST_CLASS, COST_BBOX, COST_GIOU = 1.0, 5.0, 2.0

cost_class = -pred_probs[:, gt_labels]                                   # (preds, targets) = (3, 2)
cost_bbox  = torch.cdist(pred_boxes, gt_boxes, p=1)                      # (preds, targets)
cost_giou  = -generalized_box_iou(box_cxcywh_to_xyxy(pred_boxes), box_cxcywh_to_xyxy(gt_boxes))  # (preds, targets)
C = COST_CLASS*cost_class + COST_BBOX*cost_bbox + COST_GIOU*cost_giou    # (preds, targets)

np.set_printoptions(precision=3, suppress=True)
for name, m in [("class", cost_class), ("L1", cost_bbox), ("GIoU", cost_giou), ("TOTAL", C)]:
    print(f"{name:>6} cost (rows=preds, cols=GT):\n{m.numpy()}\n")

## 3. The Hungarian algorithm, written out

We need the cheapest one-to-one assignment. Brute force is `O(n!)` — fine for our 3×2 toy,
hopeless for 100 queries. The Hungarian algorithm (Kuhn–Munkres) does it in `O(n³)`.

**The idea in one paragraph.** Keep a *potential* `u[i]` for every row and `v[j]` for every
column, maintaining the invariant `cost[i][j] - u[i] - v[j] >= 0` everywhere. Cells where that
is exactly `0` are "tight" and are the only ones we are allowed to match on. Add rows one at a
time: search for an augmenting path through tight cells; if none exists, shift the potentials by
the smallest slack `delta` — this makes at least one new cell tight without ever breaking the
invariant — and keep searching. When every row is matched, the invariant plus complementary
slackness proves the matching is optimal.

The version below is the standard `O(n³)` shortest-augmenting-path formulation.

In [ ]:
def hungarian(cost):
    """Minimum-cost perfect matching on one side of a rectangular cost matrix.

    Args:
        cost: (n, m) tensor. Rows are predictions, columns are targets.
    Returns:
        (row_idx, col_idx) LongTensors of length min(n, m); row_idx is sorted.
    """
    cost = cost.detach().cpu().double()
    n, m = cost.shape
    if n == 0 or m == 0:
        z = torch.zeros(0, dtype=torch.int64)
        return z, z
    # The algorithm below assumes at least as many columns as rows.
    if n > m:
        col_idx, row_idx = hungarian(cost.t())
        order = torch.argsort(row_idx)
        return row_idx[order], col_idx[order]

    INF = float("inf")
    # u/v are the dual potentials, one per row and per column. The invariant is
    # cost[i][j] - u[i] - v[j] >= 0 for every cell, with equality on matched cells.
    u = [0.0] * (n + 1)
    v = [0.0] * (m + 1)
    p = [0] * (m + 1)      # p[j] = row currently matched to column j (0 = none)
    way = [0] * (m + 1)    # way[j] = previous column on the augmenting path

    for i in range(1, n + 1):
        p[0] = i                       # column 0 is a sentinel holding the free row
        j0 = 0
        minv = [INF] * (m + 1)
        used = [False] * (m + 1)
        while True:                    # grow a shortest augmenting path
            used[j0] = True
            i0, delta, j1 = p[j0], INF, -1
            for j in range(1, m + 1):
                if used[j]:
                    continue
                cur = cost[i0 - 1][j - 1].item() - u[i0] - v[j]
                if cur < minv[j]:
                    minv[j], way[j] = cur, j0
                if minv[j] < delta:
                    delta, j1 = minv[j], j
            for j in range(m + 1):     # shift potentials so a new tight edge appears
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:             # reached a free column -> path complete
                break
        while j0:                      # flip the path, growing the matching by one
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1

    col_of_row = [0] * n
    for j in range(1, m + 1):
        if p[j]:
            col_of_row[p[j] - 1] = j - 1
    return torch.arange(n), torch.as_tensor(col_of_row, dtype=torch.int64)

### Does it actually work?

Two checks: brute force on small matrices, and the classic textbook example.

In [ ]:
# 1) brute force -- enumerate every one-to-one assignment and compare totals.
#    (sum in float64: a float32 sum would drift by ~1e-6 and hide nothing useful)
torch.manual_seed(0)
worst = 0.0
for trial in range(300):
    n, m = int(torch.randint(1, 6, (1,))), int(torch.randint(1, 6, (1,)))
    C_rand = (torch.randn(n, m) * 10).double()
    r, c = hungarian(C_rand)
    mine = C_rand[r, c].sum().item() if len(r) else 0.0
    k = min(n, m)
    best = min((sum(C_rand[rows[i], cols[i]].item() for i in range(k))
                for rows in itertools.permutations(range(n), k)
                for cols in itertools.permutations(range(m), k)), default=0.0)
    worst = max(worst, abs(mine - best))
print(f"300 random matrices, worst gap vs brute force: {worst:.2e}")
assert worst < 1e-9

# 2) a hand-checkable case. Every row's cheapest cell is in column 0, so a greedy
#    row-by-row choice grabs it immediately and then has to take expensive leftovers.
#    The optimum runs the other way down the anti-diagonal.
C_demo = torch.tensor([[1.0, 2.0, 3.0],
                       [2.0, 4.0, 6.0],
                       [3.0, 6.0, 9.0]])
r, c = hungarian(C_demo)
print("assignment:", list(zip(r.tolist(), c.tolist())),
      " total:", C_demo[r, c].double().sum().item())
assert C_demo[r, c].double().sum().item() == 10.0        # rows 0,1,2 -> cols 2,1,0
print("greedy row-by-row takes (0,0)=1 then (1,1)=4 then (2,2)=9  -> 14")
print("the optimum is 3+4+3 = 10, which no greedy pass would ever find")

### Back to the toy example

In [ ]:
row, col = hungarian(C)
row, col = row.tolist(), col.tolist()
print("Hungarian assignment:")
for r, c_ in zip(row, col):
    print(f"   prediction {r}  ->  ground truth {c_} ({COCO_CLASSES[gt_labels[c_]]})   cost {C[r, c_]:.3f}")
unmatched = sorted(set(range(len(pred_boxes))) - set(row))
print(f"   prediction {unmatched}  ->  no object (∅)")
print(f"\ntotal cost: {C[row, col].sum():.3f}")

# brute force check -- with only 3x2 we can enumerate every one-to-one assignment
best = min(((sum(C[p[i], i] for i in range(2)), p)
            for p in itertools.permutations(range(3), 2)), key=lambda t: t[0])
print(f"brute-force minimum: {best[0]:.3f} using preds {best[1]}  -> matches: {abs(best[0]-C[row,col].sum())<1e-5}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
im_ = ax.imshow(C.numpy(), cmap="viridis")
for i in range(C.shape[0]):
    for j in range(C.shape[1]):
        ax.text(j, i, f"{C[i,j]:.2f}", ha="center", va="center", color="w", fontsize=11)
for r, c_ in zip(row, col):
    ax.add_patch(plt.Rectangle((c_-.5, r-.5), 1, 1, fill=False, edgecolor="red", lw=4))
ax.set_xticks([0, 1], ["GT 0 (cat)", "GT 1 (remote)"])
ax.set_yticks(range(3), [f"pred {i}" for i in range(3)])
ax.set_title("cost matrix — red = chosen assignment")
plt.colorbar(im_); plt.tight_layout(); plt.show()

**Prediction 1 lost.** It was a perfectly reasonable cat box, but prediction 0 was slightly better, and the assignment is one-to-one — so prediction 1 is assigned `∅` and gets *pushed toward "no object"* by the loss.

That single mechanic is what replaces NMS. Duplicates aren't filtered after the fact; they're trained away.

## 4. Why *two* box losses? L1 alone has a scale bug

The paper: *"the most commonly-used ℓ1 loss will have different scales for small and large boxes even if their relative errors are similar"* (§3.1). Watch it happen.

In [ ]:
# two errors that are IDENTICAL in relative terms: each box is off by 10% of its own width
small_gt   = torch.tensor([[0.20, 0.20, 0.10, 0.10]])
small_pred = torch.tensor([[0.21, 0.20, 0.10, 0.10]])    # off by 0.01 = 10% of width
large_gt   = torch.tensor([[0.50, 0.50, 0.60, 0.60]])
large_pred = torch.tensor([[0.56, 0.50, 0.60, 0.60]])    # off by 0.06 = 10% of width

for name, p, g in [("small box", small_pred, small_gt), ("large box", large_pred, large_gt)]:
    l1 = torch.cdist(p, g, p=1).item()
    giou = generalized_box_iou(box_cxcywh_to_xyxy(p), box_cxcywh_to_xyxy(g)).item()
    print(f"{name}:  L1 = {l1:.4f}   GIoU = {giou:.4f}   (1-GIoU = {1-giou:.4f})")

print("\nL1 punishes the large box 6x harder for the SAME relative error.")
print("GIoU is scale-invariant, so it rates both errors almost identically.")

The paper's **Table 4** confirms both are needed: L1 alone gives 35.8 AP, GIoU alone 39.9, **both together 40.6**.

## 5. The batched matcher

The toy above handled one image. Real training runs a batch, and each image has a
different number of objects — so the cost matrix is a different shape per image and the
matching is solved per image. That is the whole of `hungarian_matcher` below.

In [ ]:
def hungarian_matcher(outputs, targets, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
    """Match each image's predictions to its ground truth, one to one.

    Args:
        outputs: {"pred_logits": (B, Q, C+1), "pred_boxes": (B, Q, 4) cxcywh in [0,1]}
        targets: list of B dicts with "labels" (n,) and "boxes" (n, 4) cxcywh
    Returns:
        list of B (pred_idx, tgt_idx) LongTensor pairs, each of length min(Q, n).
    """
    out = []
    for b, tgt in enumerate(targets):
        probs = outputs["pred_logits"][b].softmax(-1)      # (Q, C+1)
        pred_boxes = outputs["pred_boxes"][b]              # (Q, 4)
        if len(tgt["labels"]) == 0:
            z = torch.zeros(0, dtype=torch.int64)
            out.append((z, z))
            continue
        # 1 - p is the paper's class cost; the constant 1 does not change the argmin.
        c_cls = -probs[:, tgt["labels"]]                               # (Q, n)
        c_l1 = torch.cdist(pred_boxes, tgt["boxes"], p=1)              # (Q, n)
        c_giou = -generalized_box_iou(box_cxcywh_to_xyxy(pred_boxes),
                                      box_cxcywh_to_xyxy(tgt["boxes"]))
        C = cost_class * c_cls + cost_bbox * c_l1 + cost_giou * c_giou
        out.append(hungarian(C))
    return out

## 6. A real model to match against

To see the matcher on real output we need real output. So here is DETR itself — backbone,
positional encoding, transformer, prediction heads — written out in this notebook.

This is the same architecture as [`models/`](../models), just flattened into one cell with
the inference-only path kept. The proof that it is faithful is `load_state_dict(..., strict=True)`
a couple of cells down: every parameter name has to match Facebook's released checkpoint exactly,
or it raises.

Notebooks [`03`](03_backbone_shapes_and_positional_encoding.ipynb) and
[`04`](04_transformer_and_object_queries.ipynb) take these pieces apart one at a time; here we
just need a working model.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048, aux_loss=False):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(), PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)   # +1 = "no object"
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries
        self.aux_loss = aux_loss                # keep every decoder layer's prediction

    def forward(self, images, mask=None):
        """images: (B, 3, H, W). mask: (B, H, W) with True on padded pixels."""
        if mask is None:                        # a single image needs no padding
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        # hs: (num_decoder_layers, B, num_queries, hidden_dim)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)

        outputs_class = self.class_embed(hs)             # (layers, B, queries, classes+1)
        outputs_coord = self.bbox_embed(hs).sigmoid()    # (layers, B, queries, 4)
        out = {"pred_logits": outputs_class[-1], "pred_boxes": outputs_coord[-1]}
        if self.aux_loss:
            out["aux_outputs"] = [{"pred_logits": a, "pred_boxes": b}
                                  for a, b in zip(outputs_class[:-1], outputs_coord[:-1])]
        return out


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None, aux_loss=False):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR(aux_loss=aux_loss)
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep

In [ ]:
device = get_device()
model = load_pretrained_detr(device=device)     # strict=True -> names must match exactly
im = load_image("cats")

with torch.no_grad():
    outputs = model(default_transform(im).unsqueeze(0).to(device))
outputs = {k: v.cpu() for k, v in outputs.items()}

print("device:", device, "| image (W,H):", im.size)
print("pred_logits", tuple(outputs["pred_logits"].shape),
      "| pred_boxes", tuple(outputs["pred_boxes"].shape))

probs_kept, boxes_kept, _, _ = detect(model, im, threshold=0.9)
plot_results(im, probs_kept, boxes_kept, title="our from-scratch DETR, official weights")
plt.show()

### Matching 100 predictions against 4 ground-truth boxes

In [ ]:
# pretend ground truth: 2 cats + 2 remotes (roughly where they are in the photo)
targets = [{
    "labels": torch.tensor([17, 17, 75, 75]),
    "boxes": torch.tensor([[0.26, 0.54, 0.47, 0.87],
                           [0.77, 0.41, 0.46, 0.72],
                           [0.17, 0.20, 0.21, 0.10],
                           [0.55, 0.27, 0.06, 0.24]]),
}]

indices = hungarian_matcher(outputs, targets, cost_class=1, cost_bbox=5, cost_giou=2)
pred_i, tgt_i = indices[0]

print("out of 100 predictions, these 4 were matched:\n")
probs = outputs["pred_logits"][0].softmax(-1)
for p, t in zip(pred_i.tolist(), tgt_i.tolist()):
    cls = probs[p, :-1].argmax().item()
    print(f"   query {p:3d} -> GT {t} ({COCO_CLASSES[targets[0]['labels'][t]]:<7})"
          f"  model says '{COCO_CLASSES[cls]}' p={probs[p, cls]:.3f}")
print(f"\nthe other {100-len(pred_i)} queries are all assigned ∅ (no object)")

## 7. From a matching to a number

Once matched, the loss (Eq. 2) is ordinary supervised learning:

- **`loss_ce`** — cross-entropy on classes, over *all* 100 queries (matched ones get their GT class, the rest get `∅`)
- **`loss_bbox`** — L1, on matched pairs only
- **`loss_giou`** — 1 − GIoU, on matched pairs only

In [ ]:
def _permutation_index(indices, which):
    """Turn the per-image match lists into one (batch_idx, elem_idx) advanced-index pair."""
    batch_idx = torch.cat([torch.full_like(pair[which], b) for b, pair in enumerate(indices)])
    elem_idx = torch.cat([pair[which] for pair in indices])
    return batch_idx, elem_idx


def set_criterion(outputs, targets, num_classes, indices=None,
                  eos_coef=0.1, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
    """DETR's set loss (paper Eq. 2). Returns a dict of unweighted loss terms."""
    if indices is None:
        indices = hungarian_matcher(outputs, targets, cost_class, cost_bbox, cost_giou)

    src_logits = outputs["pred_logits"]                    # (B, Q, C+1)
    num_boxes = max(sum(len(t["labels"]) for t in targets), 1)

    # --- classification: every query is supervised ---------------------------
    # Unmatched queries get the no-object id (= num_classes, the last column).
    idx = _permutation_index(indices, 0)
    target_classes = torch.full(src_logits.shape[:2], num_classes,
                                dtype=torch.int64, device=src_logits.device)
    target_classes[idx] = torch.cat([t["labels"][j] for t, (_, j) in zip(targets, indices)])
    # Down-weight no-object by eos_coef, or the loss is swamped by empty slots.
    empty_weight = torch.ones(num_classes + 1, device=src_logits.device)
    empty_weight[-1] = eos_coef
    loss_ce = F.cross_entropy(src_logits.transpose(1, 2), target_classes, empty_weight)

    # --- boxes: only the matched pairs ---------------------------------------
    src_boxes = outputs["pred_boxes"][idx]
    tgt_boxes = torch.cat([t["boxes"][j] for t, (_, j) in zip(targets, indices)], dim=0)
    loss_bbox = F.l1_loss(src_boxes, tgt_boxes, reduction="none").sum() / num_boxes
    loss_giou = (1 - torch.diag(generalized_box_iou(box_cxcywh_to_xyxy(src_boxes),
                                                    box_cxcywh_to_xyxy(tgt_boxes)))).sum() / num_boxes

    # --- logging only: how many objects did the model think were there? ------
    with torch.no_grad():
        card_pred = (src_logits.argmax(-1) != num_classes).sum(1).float()
        tgt_len = torch.as_tensor([len(t["labels"]) for t in targets],
                                  dtype=torch.float, device=src_logits.device)
        cardinality_error = F.l1_loss(card_pred, tgt_len)

    return {"loss_ce": loss_ce, "loss_bbox": loss_bbox, "loss_giou": loss_giou,
            "cardinality_error": cardinality_error}

In [ ]:
losses = set_criterion(outputs, targets, num_classes=91, eos_coef=0.1)
weights = {"loss_ce": 1, "loss_bbox": 5, "loss_giou": 2}
for k, v in losses.items():
    tag = f"x{weights[k]}" if k in weights else "(logging only)"
    print(f"   {k:<18} {float(v):8.4f}   {tag}")

total = sum(losses[k] * w for k, w in weights.items())
print(f"\n   weighted total     {float(total):8.4f}")

### `eos_coef` — the class-imbalance fix

95 of 100 queries are `∅`. Left alone, cross-entropy would be dominated by "predict no-object", and the model would learn to detect nothing. DETR down-weights the `∅` class by **10×**:

```python
empty_weight = torch.ones(num_classes + 1)
empty_weight[-1] = eos_coef        # 0.1
```

In [ ]:
# what the class weights look like, and what happens without them
for eos in (1.0, 0.1):
    w = torch.ones(92); w[-1] = eos
    l = set_criterion(outputs, targets, num_classes=91, eos_coef=eos)["loss_ce"]
    print(f"eos_coef={eos:<4}  weight on ∅ = {w[-1]:.2f}   loss_ce = {float(l):.4f}")

print("\n96 of 100 queries are ∅ here. Weighting them the same as real objects lets")
print("'predict nothing' dominate the gradient -- the model learns to detect nothing.")
print("\npaper §3.1: 'we down-weight the log-probability term when c_i = ∅ by a factor 10'")

### `cardinality_error` is not a loss

In our `set_criterion` it sits inside a `torch.no_grad()` block, so it never backpropagates. It logs `|predicted object count − true count|` so you can watch the model learn *how many* things are present.

## 8. The matching is recomputed every single step

A crucial and easy-to-miss point: the assignment is **not** fixed. It's recomputed at every forward pass, for every image, and it changes as the model trains. Query 45 might own the left cat this step and nothing at all the next.

This is what makes the loss **permutation-invariant**: the model is never asked to output objects in a particular order — only to produce the right *set*.

In [ ]:
# perturb the predictions slightly and watch the assignment move
torch.manual_seed(3)
noisy = {k: v.clone() for k, v in outputs.items()}
noisy["pred_boxes"] += torch.randn_like(noisy["pred_boxes"]) * 0.05

before = hungarian_matcher(outputs, targets)[0][0].tolist()
after  = hungarian_matcher(noisy,  targets)[0][0].tolist()
print("matched query ids before noise:", sorted(before))
print("matched query ids after  noise:", sorted(after))
print("changed:", sorted(before) != sorted(after))

## What's next

`06` opens the black box: attention maps showing what the encoder and decoder actually look at. `07` puts this loss to work and trains a model from scratch.

## Exercises

**Exercise 1.** Move prediction 1 closer to the cat than prediction 0. Does the assignment flip?

<details><summary>Solution</summary>

```python
pb = pred_boxes.clone()
pb[1] = torch.tensor([0.25, 0.50, 0.30, 0.40])          # exactly on GT 0
C2 = (COST_CLASS * -pred_probs[:, gt_labels]
      + COST_BBOX * torch.cdist(pb, gt_boxes, p=1)
      + COST_GIOU * -generalized_box_iou(box_cxcywh_to_xyxy(pb), box_cxcywh_to_xyxy(gt_boxes)))
r2, c2 = hungarian(C2)
print("assignment:", list(zip(r2.tolist(), c2.tolist())))
print("total cost:", C2[r2, c2].sum().item(), "(was", C[row, col].sum().item(), ")")
```

Prediction 1 now wins GT 0 and **prediction 0 becomes ∅**. The cost drops, because the matcher always takes the globally cheapest one-to-one assignment.

The lesson: which query "owns" an object is not fixed — it's decided fresh at every training step by whichever prediction currently fits best.
</details>

---

**Exercise 2.** Set `COST_GIOU = 0`, then `COST_BBOX = 0`. Does the matching change?

<details><summary>Solution</summary>

```python
for cc, cb, cg in [(1, 5, 2), (1, 5, 0), (1, 0, 2), (1, 0, 0)]:
    Cx = cc*cost_class + cb*cost_bbox + cg*cost_giou
    r, c_ = hungarian(Cx)
    print(f"class={cc} bbox={cb} giou={cg} -> {list(zip(r.tolist(), c_.tolist()))}")
```

In this easy toy example the assignment is stable — the boxes are far enough apart that any single term picks the same pairing. Matching only becomes sensitive to the weights when candidates are genuinely close, which is the common case in real images.

Note the last row (`bbox=0, giou=0`): matching on class alone is ambiguous, since two predictions share the class "cat".
</details>

---

**Exercise 3.** What happens when there are more ground-truth objects than predictions?

<details><summary>Solution</summary>

```python
gt4 = torch.rand(4, 4).clamp(0.1, 0.4) + torch.tensor([0.2, 0.2, 0.0, 0.0])
lab4 = torch.tensor([17, 75, 17, 75])
C3 = (torch.cdist(pred_boxes, gt4, p=1) * 5
      - generalized_box_iou(box_cxcywh_to_xyxy(pred_boxes), box_cxcywh_to_xyxy(gt4)) * 2)
r3, c3 = linear_sum_assignment(C3.numpy())
print("3 predictions, 4 targets ->", len(r3), "matches")
```

You get `min(num_queries, num_targets) = 3` matches; one ground-truth object goes **undetected and unsupervised**. That's why N must comfortably exceed the object count — the paper picks N=100 for COCO where the max is 63. With N too small, DETR simply cannot represent all the objects.
</details>

---

**Exercise 4.** Set `eos_coef=1.0` and compare `loss_ce`. Which way does it move, and why?

<details><summary>Solution</summary>

```python
for eos in [0.1, 1.0]:
    crit = SetCriterion(91, matcher, {"loss_ce": 1, "loss_bbox": 5, "loss_giou": 2},
                        eos_coef=eos, losses=["labels"])
    print(f"eos_coef={eos}: loss_ce = {float(crit(outputs, targets)['loss_ce']):.4f}")
```

`loss_ce` goes **up** with `eos_coef=1.0`. The 96 unmatched queries now count 10× more toward the loss, and cross-entropy is an average over all 100 queries.

More importantly it changes what the model *learns*: with `eos_coef=1.0` the overwhelming ∅ majority dominates the gradient and the model drifts toward predicting nothing at all. Down-weighting ∅ by 10× is what keeps the rare "real object" signal audible — the same class-imbalance problem Focal Loss addresses in RetinaNet.
</details>